# **Notebook**: Landslide Exposure Database (LED) - Part 2

__Date__: September, 2025  
__Author__: D. Acosta-Reyes  
__Supervisor__: J. Wartman  
__University of Washington__

__Content__: Code utilized to aggregate information at the building-level  

See previous notebooks for:
* PART 1: Building Inventory Integration
    * Spatial Join and Nearest-Neighbor Comparison
    * Geometric filters
    * Proximity evaluation
    * Machine Learning Classification
    * Landslide Susceptibility Attribution  

__This notebook contains__:
* PART 2: Geographic Classification
    * Physiographic Divisions
    * Census Divisions
    * States
    * Census Tracts
    * Census Blocks  

See next notebook for:
* PART 3: Socioeconomic Attributes
    * Population
    * CRE
    * Poverty and Income
    * Error Metrics

### Initial setup

In [1]:
'''Load and import necessary libraries'''
# General data libraries
import numpy as np
import pandas as pd
import geopandas as gpd

import os

# Define data path
data_path = "/Users/danielacosta/Library/CloudStorage/OneDrive-UW/0 - DA General Exam/Paper 1 - Exposure analysis/Data"

#### Load datasets for geographic classification

In [ ]:
# Load final inventories as data points (footprints)
building_inventory = gpd.read_file(os.path.join(data_path, "final_overture_inventory_points.gpkg"))

# load cenus blocks
census_blocks = gpd.read_file(os.path.join(data_path, "census_gov_data/tabblock20", "tl_2025_02_tabblock20.zip"))

# load census tracts
census_tracts = gpd.read_file(os.path.join(data_path, "census_gov_data", "cb_2024_us_tract_500k.zip"))

# load census divisions
census_divisions = gpd.read_file(os.path.join(data_path, "census_gov_data", "cb_2024_us_division_500k.zip"))

# load physiographic divisions
physio_divisions = gpd.read_file(os.path.join(data_path, "census_gov_data", "physio_shp.zip"))

### A) Census Blocks  
This dataset contains baseline information on:
- GEOID20 that will help to associate all other Census boundaries.  
- Urban-rural classification

In [4]:
census_blocks.head()

,STATEFP20,COUNTYFP20,TRACTCE20,BLOCKCE20,GEOID20,GEOIDFQ20,NAME20,MTFCC20,UR20,UACE20,FUNCSTAT20,ALAND20,AWATER20,INTPTLAT20,INTPTLON20,HOUSING20,POP20,geometry
0,02,050,000100,3124,020500001003124,1000000US020500001003124,Block 3124,G5040,R,None,S,0,7676,+60.7768495,-161.4132658,0,0,"POLYGON ((-161.41397 60.77663, -161.4138 60.77..."
1,02,050,000100,3149,020500001003149,1000000US020500001003149,Block 3149,G5040,R,None,S,24236,0,+60.9116273,-161.2211055,9,56,"POLYGON ((-161.22319 60.91142, -161.22263 60.9..."
2,02,290,000300,1155,022900003001155,1000000US022900003001155,Block 1155,G5040,R,None,S,0,559665,+65.7062190,-156.4010766,0,0,"POLYGON ((-156.41178 65.70592, -156.41154 65.7..."
3,02,275,000300,1134,022750003001134,1000000US022750003001134,Block 1134,G5040,R,None,S,1387681,0,+56.0632563,-132.5463609,1,2,"POLYGON ((-132.55743 56.06108, -132.55738 56.0..."
4,02,020,001000,2035,020200010002035,1000000US020200010002035,Block 2035,G5040,U,02305,S,12140,0,+61.2111596,-149.8761254,47,69,"POLYGON ((-149.87715 61.21086, -149.87715 61.2..."


In [ ]:
# Spatial join building inventory with census blocks to get GEOID20 and Urban/Rural classification
if building_inventory.crs != census_blocks.crs:
    building_inventory = building_inventory.to_crs(census_blocks.crs)  # Ensure same CRS
building_inventory_join = gpd.sjoin(building_inventory, census_blocks[['GEOID20', 'UR20', 'geometry']], how='left', predicate='intersects')

### B) Census tracts

In [ ]:
if building_inventory_join.crs != census_tracts.crs:
    building_inventory_join = building_inventory_join.to_crs(census_tracts.crs)  # Ensure same CRS
building_inventory_join = gpd.sjoin(building_inventory_join, census_tracts[['TRACTCE','GEOIDFQ', 'geometry']], how='left', predicate='intersects')
building_inventory_join = building_inventory_join.rename(columns={'TRACTCE': 'TRACTCE20'})

### C) Census States  
Already included in STATEFP column  

### D) Census Divisions

In [10]:
census_divisions.head()

,DIVISIONCE,GEOIDFQ,GEOID,NAME,NAMELSAD,LSAD,ALAND,AWATER,geometry
0,5,0300000US5,5,South Atlantic,South Atlantic Division,69,687131859703,86332970109,"MULTIPOLYGON (((-75.56752 39.5102, -75.56477 3..."
1,3,0300000US3,3,East North Central,East North Central Division,69,629305297107,151240572209,"MULTIPOLYGON (((-82.73447 41.60351, -82.72425 ..."
2,4,0300000US4,4,West North Central,West North Central Division,69,1314697305351,33036910075,"MULTIPOLYGON (((-89.59206 47.96668, -89.59147 ..."
3,8,0300000US8,8,Mountain,Mountain Division,69,2217924639279,18695090521,"POLYGON ((-120.00645 39.27288, -120.00643 39.2..."
4,9,0300000US9,9,Pacific,Pacific Division,69,2320566048992,295497397617,"MULTIPOLYGON (((-133.26738 55.17638, -133.2634..."


In [ ]:
if building_inventory_join.crs != census_divisions.crs:
    building_inventory_join = building_inventory_join.to_crs(census_divisions.crs)  # Ensure same CRS
building_inventory_join = gpd.sjoin(building_inventory_join, census_divisions[['GEOID', 'NAME', 'geometry']], how='left', predicate='intersects')
building_inventory_join = building_inventory_join.rename(columns={'GEOID': 'census_div_id', 'NAME': 'DIVISION_NAME'})

### E) Physiographic Provinces

In [11]:
physio_divisions.head()

,AREA,PERIMETER,PHYSIODD_,PHYSIODD_I,FCODE,FENCODE,DIVISION,PROVINCE,SECTION,PROVCODE,geometry
0,40.121,36.938,2,72,122,12b,INTERIOR PLAINS,CENTRAL LOWLAND,WESTERN LAKE,12,"POLYGON ((-103.00201 49.00395, -102.94103 49.0..."
1,21.976,39.951,3,59,131,13a,INTERIOR PLAINS,GREAT PLAINS,"MISSOURI PLATEAU, GLACIATED",13,"POLYGON ((-113.55362 48.99719, -112.19278 48.9..."
2,2.706,18.014,4,33,241,24a,PACIFIC MOUNTAIN SYSTEM,PACIFIC BORDER,PUGET TROUGH,24,"POLYGON ((-122.35828 47.59034, -122.33666 47.5..."
3,3.636,8.140,5,34,231,23a,PACIFIC MOUNTAIN SYSTEM,CASCADE-SIERRA MOUNTAINS,NORTHERN CASCADE MOUNTAINS,23,"POLYGON ((-122.14882 49.00173, -120.85696 48.9..."
4,30.059,28.208,6,48,190,19,ROCKY MOUNTAIN SYSTEM,NORTHERN ROCKY MOUNTAINS,None,19,"POLYGON ((-119.67419 49.00147, -118.84358 48.9..."


In [ ]:
if building_inventory_join.crs != physio_divisions.crs:
    building_inventory_join = building_inventory_join.to_crs(physio_divisions.crs)  # Ensure same CRS
building_inventory_join = gpd.sjoin(building_inventory_join, physio_divisions[['DIVISION', 'PROVINCE', 'PROVCODE','geometry']], how='left', predicate='intersects')
building_inventory_join = building_inventory_join.rename(columns={'DIVISION': 'physio_div_id', 'PROVINCE': 'PROVINCE_NEW', 'PROVCODE': 'PROVCODE_NEW'})

In [ ]:
# save the joined inventory with census attributes to file
building_inventory_join.to_file(os.path.join(data_path, "process_building_inventory", "final_overture_inventory_points_wGeo.gpkg"), driver='GPKG')

### End of Script